# BÁO CÁO GIỮA KỲ: NHẬP MÔN PHÂN TÍCH DỮ LIỆU VÀ AI
## TÊN ĐỀ TÀI : PHÂN TÍCH VÀ DỰ ĐOÁN LOẠI TỔN THƯƠNG DA TRÊN BỘ DỮ LIỆU SKIN LESIONS

### Họ Tên: Quách Kim Anh
### MSSV: 24657701
### Lớp: DHHTTT20A

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## MỤC TIÊU ĐỀ TÀI : 
- Phân tích dữ liệu và tìm hiểu đặc điểm của các loại tổn thương da.

- Khảo sát mối liên hệ giữa tuổi, giới tính, vị trí, kích thước, triệu chứng và loại chẩn đoán.

- Xây dựng mô hình Decision Tree để dự đoán loại tổn thương da.

- Đánh giá hiệu quả dự đoán của mô hình thông qua các chỉ số Accuracy, Precision, Recall, F1-score.

- Xác định các đặc điểm quan trọng trong quá trình phân loại tổn thương da.

**Output**: Mô hình có khả năng dự đoán nhãn chẩn đoán dựa trên các thông tin đầu vào và đánh giá được hiệu quả phân loại trên tập dữ liệu kiểm tra.
## VỀ DATASET:

Dataset bao gồm 12 thuộc tính, được chia thành hai nhóm:

**1. Các biến đầu vào (Features):**
- age: Tuổi bệnh nhân.
- gender: Giới tính.
- region: Vị trí xuất hiện tổn thương da.
- diameter_1: Đường kính thứ nhất của tổn thương.
- diameter_2: Đường kính thứ hai của tổn thương.
- itch: Tình trạng ngứa.
- grew: Tình trạng phát triển của tổn thương.
- hurt: Tình trạng đau.
- changed: Tình trạng thay đổi.
- bleed: Tình trạng chảy máu.
- elevation: Tình trạng nổi cao của tổn thương.

**2. Biến mục tiêu (Target):**
- diagnostic: Loại chẩn đoán tổn thương da, được sử dụng làm nhãn để mô hình Decision Tree học và dự đoán.

Dataset được sử dụng để phân tích đặc điểm tổn thương da và xây dựng mô hình dự đoán loại tổn thương dựa trên 11 biến đầu vào.

# PHẦN 1 : TIỀN XỬ LÝ DỮ LIỆU

## 1.1. Import file và đọc file

In [ ]:
import pandas as pd
import numpy as np

# Đọc dữ liệu
df = pd.read_csv("/kaggle/input/datasets/mahdavi1202/skin-cancer/metadata.csv")
# Kiểm tra 5 dòng đầu tiên
print("5 DÒNG ĐẦU TIÊN:")
print(df.head())

# Kiểm tra kích thước dữ liệu
print("\nKÍCH THƯỚC DỮ LIỆU:")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])

# Kiểm tra tên các cột
print("\nTÊN CÁC CỘT:")
print(df.columns.tolist())

# Kiểm tra kiểu dữ liệu ban đầu
print("\nKIỂU DỮ LIỆU BAN ĐẦU:")
for col in df.columns:
    print(col, ":", df[col].dtype)

## 1.2. Kiểm tra dữ liệu thiếu và trùng

In [ ]:
# Kiểm tra số lượng dữ liệu thiếu ở mỗi cột
print("a) Dữ liệu thiếu:")
print(df.isnull().sum())
# Kiểm tra số dòng bị trùng
print("b) Số dòng trùng:", df.duplicated().sum())

## 1.3. Kiểm tra giá trị của các cột <br>
Phần này giúp mình biết dữ liệu thực tế đang có những giá trị gì trước khi chuẩn hóa.

In [ ]:
# Kiểm tra các giá trị của từng cột
print("GIÁ TRỊ CỦA CÁC CỘT:")

for col in df.columns:
    print("\n---", col, "---")
    print(df[col].value_counts(dropna=False))

## 1.4. Chuẩn hóa dữ liệu

In [ ]:
# 1. Các cột dạng số
numeric_columns = [
    'age',
    'fitspatrick',
    'diameter_1',
    'diameter_2'
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')


# 2. Các cột dạng chuỗi / phân loại
categorical_columns = [
    'patient_id',
    'background_father',
    'background_mother',
    'gender',
    'region',
    'diagnostic',
    'img_id'
]

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype('string')
        .str.strip()
        .str.upper()
    )


# 3. Các cột TRUE/FALSE/UNK
boolean_columns = [
    'smoke',
    'drink',
    'pesticide',
    'skin_cancer_history',
    'cancer_history',
    'has_piped_water',
    'has_sewage_system',
    'itch',
    'grew',
    'hurt',
    'changed',
    'bleed',
    'elevation',
    'biopsed'
]

for col in boolean_columns:
    df[col] = (
        df[col]
        .astype('string')
        .str.strip()
        .str.upper()
    )

cột lesion_id k cần đổi do nó là mã định danh và kiểu dữ liệu int64 là phù hợp vì nó không phải biến dùng để tính toán

## 1.5. Kiểm tra lại sau chuẩn hóa

In [ ]:
# Kiểm tra kiểu dữ liệu sau chuẩn hóa
print("KIỂU DỮ LIỆU SAU CHUẨN HÓA:")

for col in df.columns:
    print(f"{col:<25} {df[col].dtype}")
    
# Kiểm tra dữ liệu thiếu sau chuẩn hóa
print("\nDỮ LIỆU THIẾU SAU CHUẨN HÓA:")
print(df.isnull().sum())

# Kiểm tra dữ liệu trùng sau chuẩn hóa
print("\nDỮ LIỆU TRÙNG SAU CHUẨN HÓA:")
print("Số dòng trùng:", df.duplicated().sum())

## 1.6. Tạo biến mới

In [ ]:
df['age'].hist()

In [ ]:
bins = [0, 40, 50, 60, 70, 80, 120]
labels = ['0-40', '41-50', '51-60', '61-70', '71-80', '81+']

df['AgeGroup'] = pd.cut(
    df['age'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

print("PHÂN BỐ NHÓM TUỔI")
print(df['AgeGroup'].value_counts().sort_index())

## 1.7. Kiểm tra dữ liệu lần cuối sau khi tiền xử lí

In [ ]:
# Kiểm tra dữ liệu sau khi hoàn thành tiền xử lí
print("THÔNG TIN DỮ LIỆU SAU TIỀN XỬ LÍ:")
df.info()

print("\n 5 DÒNG DỮ LIỆU SAU TIỀN XỬ LÍ:")
print(df.head())

# PHẦN 2: PHÂN TÍCH DỮ LIỆU VÀ BIỂU ĐỒ

# 2.1. Phân bố các loại chẩn đoán
 Mục tiêu: Hiểu được sự phân bố của các loại chẩn đoán trong bộ dữ liệu.

In [ ]:
import matplotlib.pyplot as plt

# Đếm số lượng từng loại chẩn đoán
diagnostic_count = df['diagnostic'].value_counts()

# Tính tỷ lệ phần trăm
diagnostic_percent = (
    df['diagnostic']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# Tạo bảng thống kê
diagnostic_table = pd.DataFrame({
    'Số lượng': diagnostic_count,
    'Tỷ lệ (%)': diagnostic_percent
})

print("===== PHÂN BỐ CÁC LOẠI CHẨN ĐOÁN =====")
print(diagnostic_table)

# Vẽ biểu đồ số lượng các loại chẩn đoán
plt.figure(figsize=(8, 5))

diagnostic_count.plot(kind='bar',  color=['#A8DADC', '#FFD6A5', '#FFADAD',
           '#BDE0FE', '#CDEAC0', '#D8B4E2'])

plt.title('Phân bố các loại chẩn đoán')
plt.xlabel('Loại chẩn đoán')
plt.ylabel('Số lượng')
plt.xticks(rotation=0)

plt.show()

### Nhận xét

Bộ dữ liệu gồm 6 loại chẩn đoán khác nhau. BCC có số lượng mẫu cao nhất, tiếp theo là ACK. MEL có số lượng mẫu thấp nhất. Điều này cho thấy số lượng dữ liệu giữa các nhóm chẩn đoán không đồng đều.Đây là cơ sở để lựa chọn và thực hiện các phương pháp phân tích, đồng thời cần lưu ý sự chênh lệch số lượng giữa các nhóm khi xây dựng mô hình dự đoán.

# 2.2. Phân tích độ tuổi theo loại chẩn đoán
Mục tiêu: Phân tích độ tuổi và loại chẩn đoán để tìm hiểu sự khác biệt về độ tuổi giữa các nhóm

In [ ]:
age_diagnostic = df.groupby('diagnostic')['age'].agg([
    'count',
    'min',
    'max',
    'mean',
    'median'
]).round(2)

print(age_diagnostic)

age_mean = df.groupby('diagnostic')['age'].mean()

plt.figure(figsize=(8,5))

age_mean.plot(
    kind='bar',
    color=['#A8DADC', '#FFD6A5', '#FFADAD',
           '#BDE0FE', '#CDEAC0', '#D8B4E2']
)

plt.title('Tuổi trung bình theo loại chẩn đoán')
plt.xlabel('Loại chẩn đoán')
plt.ylabel('Tuổi trung bình')
plt.xticks(rotation=0)
plt.show()

### Nhận xét

Phân tích độ tuổi nhằm kiểm tra sự khác biệt về độ tuổi giữa các nhóm chẩn đoán. Kết quả giúp xác định liệu độ tuổi có sự phân bố khác nhau theo từng loại chẩn đoán hay không. Đặc điểm này cũng có thể được sử dụng làm một trong các biến đầu vào khi xây dựng mô hình dự đoán loại chẩn đoán.

# 2.3. Phân tích kích thước tổn thương theo loại chẩn đoán
Mục tiêu: Phân tích kích thước tổn thương và loại chẩn đoán để tìm hiểu sự khác biệt về kích thước giữa các nhóm.

In [ ]:
diameter_diagnostic = df.groupby('diagnostic')['AverageDiameter'].agg(['count','min','max','mean','median']).round(2)

print(diameter_diagnostic)

diameter_mean = df.groupby('diagnostic')['AverageDiameter'].mean()

plt.figure(figsize=(8,5))

diameter_mean.plot(
    kind='bar',
    color=['#A8DADC', '#FFD6A5', '#FFADAD','#BDE0FE', '#CDEAC0', '#D8B4E2'])

plt.title('Kích thước tổn thương trung bình theo loại chẩn đoán')
plt.xlabel('Loại chẩn đoán')
plt.ylabel('Kích thước trung bình')
plt.xticks(rotation=0)
plt.show()

### Nhận xét

Phân tích kích thước giúp xác định sự khác biệt về đặc điểm hình thái của tổn thương giữa các nhóm chẩn đoán. Kết quả góp phần làm rõ mối quan hệ giữa kích thước tổn thương và loại chẩn đoán. Đồng thời, các biến kích thước có thể được sử dụng làm thông tin đầu vào cho mô hình dự đoán.

# 2.4. Phân tích triệu chứng theo loại chẩn đoán
Mục tiêu: So sánh các triệu chứng của tổn thương giữa các loại chẩn đoán.

Các triệu chứng trong dataset: <br>

itch – ngứa<br>
grew – phát triển<br>
hurt – đau<br>
changed – thay đổi<br>
bleed – chảy máu<br>
elevation – nổi cao<br>

In [ ]:
symptom_columns = ['itch','grew', 'hurt','changed', 'bleed','elevation']

symptom_true = pd.DataFrame()

for symptom in symptom_columns:
    temp = df.groupby('diagnostic')[symptom].apply(
        lambda x: (x == 'TRUE').mean() * 100)

    symptom_true[symptom] = temp

symptom_true = symptom_true.round(2)

print(symptom_true)

symptom_true.plot(
    kind='bar',
    figsize=(11,6),
    color=['#A8DADC', '#FFD6A5', '#FFADAD','#BDE0FE', '#CDEAC0', '#D8B4E2'])

plt.title('Tỷ lệ xuất hiện triệu chứng theo loại chẩn đoán')
plt.xlabel('Loại chẩn đoán')
plt.ylabel('Tỷ lệ TRUE (%)')
plt.xticks(rotation=0)
plt.legend(title='Triệu chứng')
plt.show()

# 2.5. Phân tích vị trí tổn thương theo loại chẩn đoán
Mục tiêu: Phân tích vị trí xuất hiện của tổn thương và sự phân bố các loại chẩn đoán

In [ ]:
region_diagnostic = pd.crosstab(df['region'],df['diagnostic'])

print("===== SỐ LƯỢNG =====")
print(region_diagnostic)

region_percent = pd.crosstab(
    df['region'],
    df['diagnostic'],
    normalize='index'
).mul(100).round(2)

print("\n===== TỶ LỆ (%) =====")
print(region_percent)

region_percent.plot(
    kind='bar',
    figsize=(12,6),
    color=['#A8DADC', '#FFD6A5', '#FFADAD','#BDE0FE', '#CDEAC0', '#D8B4E2']
)

plt.title('Tỷ lệ các loại chẩn đoán theo vị trí tổn thương')
plt.xlabel('Vị trí tổn thương')
plt.ylabel('Tỷ lệ (%)')
plt.xticks(rotation=45)
plt.legend(title='Chẩn đoán')
plt.show()

### Nhận xét
Phân tích vị trí nhằm xác định sự phân bố của các loại chẩn đoán tại những vị trí khác nhau trên cơ thể. Kết quả giúp làm rõ mối quan hệ giữa vị trí tổn thương và loại chẩn đoán. Bên cạnh mục tiêu phân tích, vị trí tổn thương cũng là một đặc điểm có thể được sử dụng làm biến đầu vào cho bài toán dự đoán.

### Nhận xét

Các loại chẩn đoán có sự phân bố khác nhau giữa các vị trí tổn thương. Việc sử dụng tỷ lệ phần trăm giúp so sánh cơ cấu chẩn đoán giữa các vị trí, hạn chế ảnh hưởng của sự khác nhau về số lượng mẫu ở từng vị trí.

## 2.6 Bảng tổng hợp

In [ ]:
summary = df.groupby('diagnostic').agg(
    SoLuong=('diagnostic', 'count'),
    TuoiTrungBinh=('age', 'mean'),
    TuoiTrungVi=('age', 'median'),
    KichThuocTrungBinh=('AverageDiameter', 'mean'),
    DuongKinh1TrungBinh=('diameter_1', 'mean'),
    DuongKinh2TrungBinh=('diameter_2', 'mean')
).round(2)

print(summary)

### Nhận xét

Bảng tổng hợp giúp so sánh các đặc điểm chính giữa các nhóm chẩn đoán, bao gồm số lượng mẫu, độ tuổi và kích thước tổn thương. Qua đó có thể xác định những đặc điểm có sự khác biệt giữa các nhóm và lựa chọn các biến phù hợp để đưa vào mô hình dự đoán.

# 2.7. Dự đoán chẩn đoán
**Mục tiêu**
Xây dựng mô hình dự đoán loại chẩn đoán dựa trên các đặc điểm của người bệnh và tổn thương da.

## 2.7.1. Chuẩn bị dữ liệu : Dùng Decision Tree

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# BƯỚC 1: CHỌN CÁC BIẾN ĐẦU VÀO
# Các đặc điểm được sử dụng để dự đoán loại chẩn đoán
features = [
    'age',          # Tuổi
    'gender',       # Giới tính
    'region',       # Vị trí tổn thương
    'diameter_1',   # Đường kính thứ nhất
    'diameter_2',   # Đường kính thứ hai
    'itch',         # Ngứa
    'grew',         # Tổn thương phát triển
    'hurt',         # Đau
    'changed',      # Tổn thương thay đổi
    'bleed',        # Chảy máu
    'elevation'     # Nổi cao
]

# Biến mục tiêu cần dự đoán
target = 'diagnostic'

# BƯỚC 2: TẠO DATAFRAME CHO MÔ HÌNH
# Chỉ lấy các biến đầu vào và biến mục tiêu
model_df = df[features + [target]].copy()

# BƯỚC 3: XỬ LÝ DỮ LIỆU THIẾU
# Loại bỏ những dòng bị thiếu dữ liệu
# trong các cột được sử dụng cho mô hình
model_df = model_df.dropna()


# BƯỚC 4: TÁCH BIẾN ĐẦU VÀO VÀ BIẾN MỤC TIÊTIÊU
# X chứa các đặc điểm dùng để dự đoán
X = model_df[features]

# y chứa loại chẩn đoán cần dự đoán
y = model_df[target]


# BƯỚC 5: KIỂM TRA DỮ LIỆU
# In số lượng mẫu còn lại sau khi xử lý dữ liệu thiếu
print("Số mẫu dùng để xây dựng mô hình:", len(model_df))

# In số lượng mẫu của từng loại chẩn đoán
print("\nPhân bố các loại chẩn đoán:")
print(y.value_counts())

## 2.7.2. Chia dữ liệu Train/Test

In [ ]:
# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
X_train, X_test, y_train, y_test = train_test_split(
X,                  # Các biến đầu vào
y,                  # Biến mục tiêu
test_size=0.2,      # 20% dữ liệu dùng để kiểm tra
random_state=42,    # Cố định kết quả mỗi lần chạy
stratify=y          # Giữ tỷ lệ các nhóm chẩn đoán tương đối đồng đều
)

# Kiểm tra số lượng mẫu của từng tập
print("Số mẫu tập Train:", len(X_train))
print("Số mẫu tập Test:", len(X_test))

# Kiểm tra phân bố chẩn đoán trong tập Train
print("\nPhân bố chẩn đoán trong tập Train:")
print(y_train.value_counts())

# Kiểm tra phân bố chẩn đoán trong tập Test
print("\nPhân bố chẩn đoán trong tập Test:")
print(y_test.value_counts())

test_size=0.2 -> lấy 20% dữ liệu làm Test, còn 80% làm Train.
random_state=1 -> giúp mỗi lần chạy lại đều chia dữ liệu theo cùng một cách.
stratify=y -> quan trọng vì dataset của bạn có 6 nhóm chẩn đoán không đồng đều. Tham số này giúp tỷ lệ các nhóm trong Train và Test được giữ tương đối giống nhau.

### Nhận xét :
Dữ liệu được chia thành 80% tập huấn luyện và 20% tập kiểm tra. Việc sử dụng stratify giúp duy trì tương đối tỷ lệ của các nhóm chẩn đoán ở cả hai tập, phù hợp với dữ liệu có số lượng mẫu giữa các nhóm không đồng đều.

## 2.7.3. Mã hóa biến phân loại

Tại sao cần làm bước này?

Decision Tree không thể trực tiếp xử lý các giá trị dạng chữ như: MALE, FEMALE, FACE,..v..v
Vì vậy cần chuyển các biến phân loại thành dạng số.

In [ ]:
# Các biến dạng phân loại
categorical_features = [
    'gender',       # Giới tính
    'region',       # Vị trí tổn thương
    'itch',         # Ngứa
    'grew',         # Phát triển
    'hurt',         # Đau
    'changed',      # Thay đổi
    'bleed',        # Chảy máu
    'elevation'     # Nổi cao
]

# Các biến dạng số
numeric_features = [
    'age',          # Tuổi
    'diameter_1',   # Đường kính 1
    'diameter_2'    # Đường kính 2
]


# Tạo bộ xử lý dữ liệu
preprocessor = ColumnTransformer(
    transformers=[

        # One-Hot Encoding cho các biến phân loại
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),

        # Giữ nguyên các biến số
        (
            'numeric',
            'passthrough',
            numeric_features
        )
    ]
)

Các biến phân loại được mã hóa bằng One-Hot Encoding để chuyển dữ liệu dạng chữ thành dạng số mà mô hình có thể xử lý. Các biến số như tuổi và kích thước tổn thương được giữ nguyên. Bước này giúp chuẩn bị dữ liệu phù hợp cho mô hình dự đoán.

# 2.7.4. Xây dựng mô hình Decision Tree

In [ ]:
# Tạo mô hình gồm 2 bước:
# 1. Tiền xử lý dữ liệu
# 2. Huấn luyện Decision Tree
   # Bước 1: xử lý các biến đầu vào: 'preprocessor', preprocessor)
# Bước 2: xây dựng cây quyết định('classifier', DecisionTreeClassifier(
            #random_state=42,
            #max_depth=5
 
model = Pipeline(
    steps=[('preprocessor', preprocessor),('classifier', DecisionTreeClassifier(
            random_state=1,#random
            max_depth=5 # giới hạn độ sâu cây 
        ))
    ]
)

# Huấn luyện mô hình bằng tập Train
model.fit(X_train, y_train)

# Sử dụng mô hình để dự đoán tập Test
y_pred = model.predict(X_test)


print("Đã hoàn thành huấn luyện và dự đoán.")

# 2.7.5. Đánh giá mô hình 
Mục đích: 
Sau khi dự đoán xong, cần kiểm tra: Mô hình dự đoán có chính xác không?

In [ ]:
# Tính Accuracy
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("ĐỘ CHÍNH XÁC")
print("Accuracy:", round(accuracy, 4))

# In các chỉ số Precision, Recall, F1-score
print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        y_test,
        y_pred
    )
)

### Nhận xét :
Sau khi xây dựng mô hình Decision Tree, mô hình được đánh giá trên tập dữ liệu kiểm tra gồm 299 mẫu. Kết quả cho thấy mô hình đạt Accuracy = 66.89%, nghĩa là mô hình dự đoán đúng khoảng 66.89% số mẫu trong tập kiểm tra.

Bên cạnh Accuracy, các chỉ số Precision, Recall và F1-score được sử dụng để đánh giá khả năng dự đoán của mô hình trên từng loại chẩn đoán. Kết quả cho thấy khả năng dự đoán giữa các nhóm không đồng đều. Mô hình có khả năng nhận diện nhóm BCC tương đối tốt với Recall 89%, trong khi khả năng nhận diện một số nhóm như SCC và SEK còn thấp.

Cụ thể, nhóm SCC có Recall 8%, còn nhóm SEK có Recall 0%, cho thấy mô hình bỏ sót nhiều mẫu thuộc hai nhóm này. Đối với nhóm MEL, mặc dù Precision đạt 75%, Recall chỉ đạt 30%, cho thấy mô hình vẫn còn hạn chế trong việc nhận diện đầy đủ các mẫu thuộc nhóm này.

Nhìn chung, mô hình Decision Tree bước đầu có khả năng phân loại các loại tổn thương da, nhưng hiệu quả dự đoán chưa đồng đều giữa các nhóm. Kết quả này có thể chịu ảnh hưởng bởi sự mất cân bằng số lượng mẫu giữa các lớp, khi BCC và ACK có nhiều mẫu hơn trong khi MEL và SEK có số lượng mẫu ít hơn. Vì vậy, mô hình cần được cải thiện nếu muốn tăng khả năng nhận diện các nhóm có ít dữ liệu.

# PHẦN 3. KẾT LUẬN

Mô hình Decision Tree được xây dựng nhằm dự đoán loại chẩn đoán của tổn thương da dựa trên các đặc điểm có trong bộ dữ liệu. Kết quả đánh giá trên tập kiểm tra cho thấy mô hình đạt Accuracy 66.89%.

Mô hình có khả năng nhận diện một số nhóm tương đối tốt, đặc biệt là BCC với Recall 89%. Tuy nhiên, khả năng nhận diện giữa các nhóm chưa đồng đều, trong đó SCC và SEK có Recall thấp. Nguyên nhân có thể liên quan đến sự chênh lệch số lượng mẫu giữa các nhóm chẩn đoán.

Như vậy, mô hình Decision Tree đã bước đầu thực hiện được nhiệm vụ phân loại các loại tổn thương da, nhưng vẫn còn hạn chế về khả năng nhận diện các lớp có ít dữ liệu. Kết quả mô hình có thể được sử dụng làm cơ sở cho các bước cải thiện mô hình trong tương lai, chẳng hạn như cân bằng dữ liệu hoặc thử nghiệm các thuật toán phân loại khác.